In [ ]:
import pandas as pd
from blocksnet.config import log_config

log_config.set_logger_level('WARNING')

# Оценка стоимости земли проекта

In [ ]:
baseline_prepared_blocks = pd.read_pickle('../data/baseline_prepared_blocks.pickle')
baseline_prepared_blocks.head()

In [ ]:
scenario_prepared_blocks = pd.read_pickle('../data/scenario_prepared_blocks.pickle')
scenario_prepared_blocks.head()

### Инструкция по расчёту стоимости земли

Чтобы получить **цены на землю _до_ и _после застройки_**,  
нужно `запустить метод два раза` с разными входными данными:

1. **Первый запуск** — подаётся `исходная территория`
   (до зонирования и застройки).  
   → результат: `land_value` — стоимость **до застройки**.

2. **Второй запуск** — подаётся  `территория с новым зонированием и застройкой`.  
   → результат: `land_value` — стоимость **после застройки**.

3. Затем `объединяем оба результата` в один файл через метод `transfer_baseline_prices`

Так можно сравнить **изменение общей стоимости** и оценить  
влияние проекта на цену земли. Каждый файл сохраняется отдельно.

In [ ]:
from catboost import CatBoostRegressor

from urbanomy.methods.land_value_modeling import LandPriceEstimator

model = CatBoostRegressor()
model.load_model('../data/catboost_model.cbm')  # модель на лог-цене

estimator = LandPriceEstimator(
    model=model,
    blocks=baseline_prepared_blocks, 
)
baseline_pred = estimator.predict()
baseline_pred.head()


In [ ]:
baseline_pred.to_pickle('../data/baseline_blocks_value.pickle')

In [ ]:
from urbanomy.methods.land_value_modeling import LandPriceEstimator

estimator = LandPriceEstimator(
    model=model,
    blocks=scenario_prepared_blocks, #or baseline_blocks
)
scenario_pred = estimator.predict()
scenario_pred.head()

In [ ]:
scenario_pred.to_pickle('../data/scenario_blocks_value.pickle')

# Изначальная цена проектных участков с учетом нового зонирования

In [ ]:
import geopandas as gpd
import pandas as pd

from urbanomy.methods.land_value_modeling import transfer_baseline_prices

# загрузка
after = pd.read_pickle('../data/scenario_blocks_value.pickle').to_crs(32636)
before = pd.read_pickle('../data/baseline_blocks_value.pickle').to_crs(32636)

blocks_full_value = transfer_baseline_prices(after, before)

blocks_full_value.to_pickle('../data/blocks_full_value.pickle')
# фильтруем только сценарные кварталы

scn_blocks = blocks_full_value.loc[blocks_full_value['is_project'] == True].copy()


sum_before = scn_blocks['land_value_before'].sum()
sum_after  = scn_blocks['land_value'].sum()

print(f"Сценарные кварталы — суммарная стоимость ДО застройки: {sum_before:,.0f} руб.")
print(f"Сценарные кварталы — суммарная стоимость ПОСЛЕ застройки: {sum_after:,.0f} руб.")


In [ ]:
blocks_full_value.head()

# Визуализация стоимости земли

In [ ]:
from urbanomy.methods.land_value_modeling import plot_land_price_maps

visualization_output = plot_land_price_maps(
    blocks_pred=blocks_full_value,
    price_column="land_value",  # используйте "land_value_before" для цен до застройки
    buffer_radius_m=2000,
)
